In [1]:
#导入工具包
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
#第一步:图片预处理
#统一图片大小、转格式、标准化
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),   #把所有图片缩放到 128*128
    transforms.RandomHorizontalFlip(),#随机左右翻转图片（数据增强）
    transforms.ToTensor(),           #图片转为模型能识别的张量
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) # 简单归一化
])
#测试集不用翻转，只做基础处理
test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

In [3]:
#第二步：加载数据集
#加载训练集
train_data = datasets.ImageFolder(root="./dataset/train", transform=train_transform)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)

#加载test1作为训练时的验证集（每轮跑完测一次）
val_data = datasets.ImageFolder(root="./dataset/test1", transform=test_transform)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False)

#打印标签对应关系：文件夹名 → 数字
print("类别对应：", train_data.class_to_idx)

类别对应： {'blue': 0, 'red': 1, 'yellow': 2}


In [4]:
#第三步：搭建神经网络
class ConeModel(nn.Module):
    def __init__(self):
        super(ConeModel, self).__init__()
        #卷积+池化层：一共8层（4个卷积+4个池化）
        #第1组：卷积1+池化1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        #输入：[批次, 3通道, 128, 128]  输出：[批次, 16通道, 128, 128]
        self.pool1 = nn.MaxPool2d(2, 2)
        #输入：[批次, 16, 128, 128]  输出：[批次, 16, 64, 64]

        #第2组：卷积2 + 池化2
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        #输入：[批次, 16, 64, 64]  输出：[批次, 32, 64, 64]
        self.pool2 = nn.MaxPool2d(2, 2)
        #输入：[批次, 32, 64, 64]  输出：[批次, 32, 32, 32]

        #第3组：卷积3 + 池化3
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        #输入：[批次, 32, 32, 32]  输出：[批次, 64, 32, 32]
        self.pool3 = nn.MaxPool2d(2, 2)
        #输入：[批次, 64, 32, 32]  输出：[批次, 64, 16, 16]

        #第4组：卷积4 + 池化4
        self.conv4 = nn.Conv2d(64, 128, 3, padding=1)
        #输入：[批次, 64, 16, 16]  输出：[批次, 128, 16, 16]
        self.pool4 = nn.MaxPool2d(2, 2)
        #输入：[批次, 128, 16, 16]  输出：[批次, 128, 8, 8]

        #全连接层：一共 2 层
        self.fc1 = nn.Linear(128 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 3)  #最终输出3个类别：红、黄、蓝

        #激活函数
        self.relu = nn.ReLU()

    #前向传播：数据在网络里怎么走
    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = self.pool3(self.relu(self.conv3(x)))
        x = self.pool4(self.relu(self.conv4(x)))

        #把多维图片特征拉成一维向量，给全连接层使用
        x = x.view(x.size(0), -1)

        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [5]:
#第四步：训练基础配置
#自动使用GPU/CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ConeModel().to(device)

#损失函数、优化器
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

#训练轮数
epoch_num = 10

In [6]:
#第五步：开始训练 + 打印日志 
print("===== 开始训练 =====")
for epoch in range(epoch_num):
    #切换为训练模式
    model.train()
    train_loss = 0

    #遍历所有训练图片
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        #模型预测
        outputs = model(images)
        loss = loss_func(outputs, labels)

        #梯度清零、反向传播、更新参数
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss = train_loss + loss.item()

    #计算本轮平均训练损失
    avg_train_loss = train_loss / len(train_loader)
    print(f"Train Epoch : {epoch+1}   Loss : {avg_train_loss:.7f}")

     #每轮训练完，在test1上验证
    model.eval()  #切换为测试模式
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  #测试时不计算梯度，提速
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = loss_func(outputs, labels)
            val_loss = val_loss + loss.item()

            #统计预测正确数量
            _, pred = torch.max(outputs, 1)
            total = total + labels.size(0)
            correct = correct + (pred == labels).sum().item()

    avg_val_loss = val_loss / len(val_loader)
    acc = 100 * correct / total
    print(f"Test -- Average Loss : {avg_val_loss:.4f}, Accuracy : {acc:.3f} %")
    print("----------------------------------------")

===== 开始训练 =====
Train Epoch : 1   Loss : 0.8084059
Test -- Average Loss : 0.4489, Accuracy : 83.410 %
----------------------------------------
Train Epoch : 2   Loss : 0.3274303
Test -- Average Loss : 0.2839, Accuracy : 93.548 %
----------------------------------------
Train Epoch : 3   Loss : 0.1574339
Test -- Average Loss : 0.2323, Accuracy : 95.853 %
----------------------------------------
Train Epoch : 4   Loss : 0.1313765
Test -- Average Loss : 0.1469, Accuracy : 94.931 %
----------------------------------------
Train Epoch : 5   Loss : 0.1010680
Test -- Average Loss : 0.1123, Accuracy : 95.853 %
----------------------------------------
Train Epoch : 6   Loss : 0.0623791
Test -- Average Loss : 0.1411, Accuracy : 96.313 %
----------------------------------------
Train Epoch : 7   Loss : 0.0412877
Test -- Average Loss : 0.0855, Accuracy : 98.157 %
----------------------------------------
Train Epoch : 8   Loss : 0.0216184
Test -- Average Loss : 0.0479, Accuracy : 98.618 %
--------

In [7]:
#保存训练好的模型
torch.save(model.state_dict(), "cone_model.pth")
print("训练结束，模型已保存！")

训练结束，模型已保存！
